In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math 
from dataclasses import dataclass

# DeepSeek 简单demo

## 专家里的 SiLU

`ACT2FN` 是 Hugging Face 用激活函数名查找实现的表。例如模型配置写 `hidden_act="silu"`，通用模型代码就可以从表中选择 SiLU。这份教学示例固定使用 SiLU，直接写 `F.silu(...)` 就够了，也省去配置字段和 Transformers 依赖。

PyTorch 的 SiLU 逐元素计算

$$
\operatorname{SiLU}(x)=x\,\sigma(x)=\frac{x}{1+e^{-x}}.
$$

用 PyTorch 写出等价表达式就是 `x * torch.sigmoid(x)`。专家使用的是 **SwiGLU** 结构，先算 `SiLU(gate_proj(x))`，再与 `up_proj(x)` 逐元素相乘，最后通过 `down_proj` 投回原维度。这里的 gate 是专家 MLP 内的门控，路由器的 MoE gate 则负责选专家。 [PyTorch SiLU 文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.silu.html)

In [ ]:
# Experts
class DeepseekV2MLP(nn.Module):
    def __init__(self, config, hidden_size=None, intermediate_size=None):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size if hidden_size is None else hidden_size
        self.intermediate_size = (
            config.intermediate_size if intermediate_size is None else intermediate_size
        )

        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=False)

    def forward(self, x):
        gate = F.silu(self.gate_proj(x))
        return self.down_proj(gate * self.up_proj(x))

## 分组限制路由怎么选专家

对每个 token，路由器还是先给**所有路由专家**打分。普通 `greedy` 直接在全部专家里选 Top-K。`group_limited_greedy` 多做一步筛组。

1. 把专家分数从 `[token 数, 专家数]` 改成 `[token 数, 组数, 每组专家数]`。DeepSeek-V2 用**组内最高专家分数**作为组分数。
2. 每个 token 单独选分数最高的 `topk_group` 个组。布尔 mask 在专家维广播，被排除组的分数设为 `-inf`。
3. 展平回专家轴，从允许的专家里选 `top_k` 个。归一化时只处理最终选中的权重。

![普通 Top-2 从六个专家中选 E0、E2；分组 Top-2 先保留 G0，再选 E0、E1](figures/group-limited-routing.svg)

例如三组分别是 E0/E1、E2/E3、E4/E5，最高分分别是 0.35、0.30、0.08。只保留一组时就选第一组，最后用 E0、E1。普通 Top-2 会选 E0、E2。**组限制约束的是一条 token 会跨多少组，最终仍执行 `top_k` 次专家计算。** 单设备示例里的组只是索引划分。在专家并行中，如果每组放在一台设备上，这个限制就能约束单个 token 的通信扇出。全专家打分和选组仍有开销。 [DeepSeek-V2 论文 §2.2.2](https://arxiv.org/html/2405.04434) · [官方路由代码](https://huggingface.co/deepseek-ai/DeepSeek-V2/blob/main/modeling_deepseek.py#L657)

In [ ]:
# Gates Network

class MoEGate(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.top_k = config.num_experts_per_tok
        self.n_routed_experts = config.n_routed_experts
        self.routed_scaling_factor = config.routed_scaling_factor
        self.scoring_func = config.scoring_func
        self.topk_method = config.topk_method
        self.n_group = config.n_group
        self.topk_group = config.topk_group
        self.norm_topk_prob = config.norm_topk_prob
        
        # 静态化推理配置（假设配置固定）
        self.inference_norm = self.norm_topk_prob and (self.top_k > 1)
        self.use_group_limited = self.topk_method == "group_limited_greedy"
        if self.use_group_limited:
            if self.n_group < 1 or self.n_routed_experts % self.n_group:
                raise ValueError("路由专家必须能等分到各组")
            self.experts_per_group = self.n_routed_experts // self.n_group
            if not 1 <= self.topk_group <= self.n_group:
                raise ValueError("topk_group 必须在 1 到 n_group 之间")
            if not 1 <= self.top_k <= self.topk_group * self.experts_per_group:
                raise ValueError("保留组的专家数必须足够选出 top_k 个专家")

        # 门控权重
        self.gating_dim = config.hidden_size
        self.weight = nn.Parameter(torch.empty((self.n_routed_experts, self.gating_dim)))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    @torch.inference_mode()  # 禁用梯度与训练逻辑
    def forward(self, hidden_states):
        bsz, seq_len, h = hidden_states.shape
        hidden_states = hidden_states.reshape(-1, h)
        
        # 门控分数计算（保持原始数据类型）
        logits = F.linear(hidden_states, self.weight)  # [n_tokens, n_experts]
        scores = logits.softmax(dim=-1)  # 自动推断 dtype

        # Top-K 选择（静态分支）
        if self.use_group_limited:
            # 每个 token 独立选组。组分数取组内最高的专家分数。
            grouped = scores.reshape(bsz * seq_len, self.n_group, self.experts_per_group)
            group_scores = grouped.max(dim=-1).values  # [token 数, 组数]
            group_idx = group_scores.topk(self.topk_group, dim=-1, sorted=False).indices
            group_mask = torch.zeros_like(group_scores, dtype=torch.bool)
            group_mask.scatter_(1, group_idx, True)
            # 每组共用一个布尔值。未入选的组不能参与下一次 top-k。
            scores = grouped.masked_fill(~group_mask.unsqueeze(-1), -torch.inf).flatten(1)
        
        topk_weight, topk_idx = torch.topk(scores, k=self.top_k, dim=-1, sorted=False)

        # 权重归一化（静态分支）
        if self.inference_norm:
            topk_weight = topk_weight / (topk_weight.sum(dim=-1, keepdim=True) + 1e-20)
        else:
            topk_weight = topk_weight * self.routed_scaling_factor

        return topk_idx, topk_weight, None  # aux_loss 始终为 None

@dataclass
class DeepseekV2Config:
    # 1, Position Config
    max_position_embeddings: int = 163840
    vocab_size: int = 102400

    # 2, MLA Config
    # down_linear config
    q_lora_rank: int = 1536
    kv_lora_rank: int = 512

    # head_dim、heads and hidden_size config
    v_head_dim: int = 128
    qk_nope_head_dim: int = 128
    qk_rope_head_dim: int = 64
    hidden_size: int = 5120
    num_attention_heads: int = 128
    num_key_value_heads: int = 128
    
    attention_bias: bool = False

    attention_dropout: float = 0.1
    # rope config
    rope_theta: float = 10000

    # 3, MOE Config
    n_group: int = 8
    n_routed_experts: int = 160
    intermediate_size: int = 12288
    moe_intermediate_size: int = 1536
    n_shared_experts: int | None = 2
    num_experts_per_tok: int = 6
    topk_group: int = 3
    routed_scaling_factor: float = 1.0
    scoring_func: str="softmax"
    topk_method: str="greedy"
    norm_topk_prob: bool = True


In [ ]:
# 用人为设定的分数对照两条实际的路由分支。
scores = torch.tensor([0.35, 0.04, 0.30, 0.19, 0.08, 0.04])
token = torch.ones(1, 1, 1)  # [B, T, hidden_size]
for method in ("greedy", "group_limited_greedy"):
    config = DeepseekV2Config(
        hidden_size=1, n_routed_experts=6, n_group=3,
        num_experts_per_tok=2, topk_group=1, topk_method=method,
    )
    router = MoEGate(config)
    with torch.no_grad():
        # 输入是 1，logits 就等于 log(scores)，softmax 后还原指定分数。
        router.weight.copy_(scores.log().unsqueeze(-1))
    ids, weights, _ = router(token)
    print(method, "专家", ids.tolist(), "归一化权重", weights.tolist())
    expected = {0, 2} if method == "greedy" else {0, 1}
    assert set(ids[0].tolist()) == expected
    torch.testing.assert_close(weights.sum(dim=-1), torch.ones(1))

## 同一个 token 怎么交给多个专家？

假设两个 token 是 A、B，`top_k=2`，路由器给出的专家 ID 如下。

| token | 第 0 个路由槽 | 第 1 个路由槽 |
| --- | --- | --- |
| A | E2 | E0 |
| B | E1 | E2 |

**路由槽**就是一个 token 的一次专家分配。把专家 ID 展平会得到 `[2, 0, 1, 2]`。如果也想用同样的展平顺序存输入，就把 `[A, B]` 用 `repeat_interleave(2, dim=0)` 变成 `[A, A, B, B]`。它是把**每一行连续复制两次**，所以两个展平数组逐行对齐。展平后的路由槽 `r` 对应原 token 的行号是 `r // top_k`。

**训练和推理里，只要 `top_k=2`，A 就都要经过两个不同的专家。** 区别在于怎么把输入交给它们。训练参考实现先铺开所有输入再按专家分发。推理可以直接索引原始 token 行，按专家取出自己的子批次，再把加权输出累加回原 token 行。这样可以跳过一份全量的 `[token 数 × top_k, hidden_size]` 输入副本。专家计算仍发生了两次。DeepSeek-V2 官方推理实现按专家排序槽位，然后用 `r // top_k` 取行。本页用 `torch.where` 直接得到等价的 token 行号和槽号。 [DeepSeek-V2 官方实现](https://huggingface.co/deepseek-ai/DeepSeek-V2/blob/main/modeling_deepseek.py#L784) · [PyTorch repeat_interleave](https://docs.pytorch.org/docs/stable/generated/torch.repeat_interleave.html)

In [ ]:
# 为了单元测试，模拟不使用分布式（ep_size默认为1）
class DeepseekV2MoE(nn.Module):
    """
    A mixed expert module containing shared experts.
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_experts_per_tok = config.num_experts_per_tok

        self.experts = nn.ModuleList(
            [
                DeepseekV2MLP(
                    config, intermediate_size=config.moe_intermediate_size
                )
                for i in range(config.n_routed_experts)
            ]
        )
        self.gate = MoEGate(config)
        if config.n_shared_experts is not None:
            intermediate_size = config.moe_intermediate_size * config.n_shared_experts
            self.shared_experts = DeepseekV2MLP(config, intermediate_size=intermediate_size)

    @torch.inference_mode()
    def forward(self, hidden_states):
        topk_ids, topk_weight, _ = self.gate(hidden_states)
        x = hidden_states.reshape(-1, hidden_states.shape[-1])
        output = self.moe_infer(x, topk_ids, topk_weight).reshape_as(hidden_states)
        if self.config.n_shared_experts is not None:
            output = output + self.shared_experts(hidden_states)
        return output

    # 单设备推理示例。按专家取原 token，再累加回原 token 行。
    @torch.no_grad()
    def moe_infer(self, x, topk_ids, topk_weight):
        output = torch.zeros_like(x)  # [token 数, hidden_size]
        for expert_id, expert in enumerate(self.experts):
            token_ids, slots = torch.where(topk_ids == expert_id)
            if token_ids.numel() == 0:
                continue
            expert_output = expert(x[token_ids])
            weights = topk_weight[token_ids, slots].unsqueeze(-1)
            output.index_add_(0, token_ids, (expert_output * weights).to(output.dtype))
        return output

In [ ]:
# 对照两条等价的 dispatch 路径，使用上一节表格里的专家 ID。
torch.manual_seed(7)
config = DeepseekV2Config(
    hidden_size=2, n_routed_experts=3, num_experts_per_tok=2,
    moe_intermediate_size=4, n_shared_experts=1,
)
moe = DeepseekV2MoE(config).eval()
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # 两个 token，hidden_size=2
topk_ids = torch.tensor([[2, 0], [1, 2]])
topk_weight = torch.tensor([[0.75, 0.25], [0.60, 0.40]])

flat_experts = topk_ids.flatten()
repeated_x = x.repeat_interleave(2, dim=0)
print("展平后的专家", flat_experts.tolist())
print("逐行重复后的 token", repeated_x.tolist())

with torch.inference_mode():
    # 参考版先铺开 [A, A, B, B]，按槽位写回输出，再加权合并。
    by_slot = torch.empty_like(repeated_x)
    for expert_id, expert in enumerate(moe.experts):
        mask = flat_experts == expert_id
        by_slot[mask] = expert(repeated_x[mask])
    expected = (by_slot.reshape(2, 2, -1) * topk_weight.unsqueeze(-1)).sum(dim=1)
    torch.testing.assert_close(moe.moe_infer(x, topk_ids, topk_weight), expected)
    assert moe(x.unsqueeze(0)).shape == (1, 2, 2)

sample = torch.tensor([-2.0, 0.0, 2.0])
torch.testing.assert_close(F.silu(sample), sample * torch.sigmoid(sample))